In [46]:
import os
import sys
import glob
import awkward as ak
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
import torch.optim as optim
from torch.utils.data import ConcatDataset
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool
from torch_geometric.nn import GCNConv as gcn
from torch_geometric.nn import EdgeConv 
from torch_geometric.nn import DynamicEdgeConv 
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

In [47]:
sys.path.append("modules")
import architectures as arch

In [48]:
SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)

device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_gpus = torch.cuda.device_count()
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"Using device: {device} ({num_gpus} GPUs available)")

Using device: cuda (1 GPUs available)


In [49]:
#load models
DYNAMIC_MODEL_PATH   = "/standard/ldmxuva/gnn_files/run_output/dynamic_run_extended_best_model.pt"
STATIC_MODEL_PATH    = "/standard/ldmxuva/gnn_files/run_output/static_run_extended_best_model.pt"

trained_model_state_dict_dynamic = torch.load(DYNAMIC_MODEL_PATH, weights_only=True)
trained_model_state_dict_static  = torch.load(STATIC_MODEL_PATH,  weights_only=True)

model_preloaded_dynamic = arch.GNN_v3_dynamic(
    in_channels=4,
    hc1=10, hc2=20, hc3=40, hc4=50,
    fc1=25, fc2=12, fc3=6,
    k1=33,  k2=25,  k3=17, k4=9,
    out_channels=2,
)
model_preloaded_static = arch.GNN_v3_static(
    in_channels=4,
    hc1=10, hc2=20, hc3=40, hc4=50,
    fc1=25, fc2=12, fc3=6,
    out_channels=2,
)

model_preloaded_static.load_state_dict(trained_model_state_dict_static)
model_preloaded_static = model_preloaded_static.to(device)
model_preloaded_static.eval()

model_preloaded_dynamic.load_state_dict(trained_model_state_dict_dynamic)
model_preloaded_dynamic = model_preloaded_dynamic.to(device)
model_preloaded_dynamic.eval()

GNN_v3_dynamic(
  (ECal_branch): DynamicEdgeConvBlock(
    (mlp1): Sequential(
      (0): Linear(in_features=8, out_features=10, bias=True)
      (1): ReLU()
      (2): Linear(in_features=10, out_features=10, bias=True)
    )
    (mlp2): Sequential(
      (0): Linear(in_features=20, out_features=20, bias=True)
      (1): ReLU()
      (2): Linear(in_features=20, out_features=20, bias=True)
    )
    (mlp3): Sequential(
      (0): Linear(in_features=40, out_features=40, bias=True)
      (1): ReLU()
      (2): Linear(in_features=40, out_features=40, bias=True)
    )
    (mlp4): Sequential(
      (0): Linear(in_features=80, out_features=50, bias=True)
      (1): ReLU()
      (2): Linear(in_features=50, out_features=50, bias=True)
    )
    (edgeconv1): DynamicEdgeConv(nn=Sequential(
      (0): Linear(in_features=8, out_features=10, bias=True)
      (1): ReLU()
      (2): Linear(in_features=10, out_features=10, bias=True)
    ), k=33)
    (edgeconv2): DynamicEdgeConv(nn=Sequential(
      (0

In [50]:
VALIDATION_DIRECTORY = "/standard/ldmxuva/gnn_files/note_stuff/validation_graphs_all"

validation_paths = glob.glob(os.path.join(VALIDATION_DIRECTORY, "v*"))
print("Validation paths found:", validation_paths)

print("Beginning to load validation files")
val_file_list = []
for i, path in enumerate(validation_paths):
    if i > 3:
        break
    val_file_list.append(torch.load(path, weights_only=False))
    print(f"Loaded: {path}")

Validation paths found: ['/standard/ldmxuva/gnn_files/note_stuff/validation_graphs_all/validation_signal_signal_50_graphs.pt', '/standard/ldmxuva/gnn_files/note_stuff/validation_graphs_all/validation_signal_signal_5_graphs.pt', '/standard/ldmxuva/gnn_files/note_stuff/validation_graphs_all/validation_signal_signal_100_graphs.pt', '/standard/ldmxuva/gnn_files/note_stuff/validation_graphs_all/validation_signal_signal_10_graphs.pt', '/standard/ldmxuva/gnn_files/note_stuff/validation_graphs_all/validation_enriched_merged.pt']
Beginning to load validation files
Loaded: /standard/ldmxuva/gnn_files/note_stuff/validation_graphs_all/validation_signal_signal_50_graphs.pt
Loaded: /standard/ldmxuva/gnn_files/note_stuff/validation_graphs_all/validation_signal_signal_5_graphs.pt
Loaded: /standard/ldmxuva/gnn_files/note_stuff/validation_graphs_all/validation_signal_signal_100_graphs.pt
Loaded: /standard/ldmxuva/gnn_files/note_stuff/validation_graphs_all/validation_signal_signal_10_graphs.pt


In [51]:
#loaders:
loaders_list = []
loader_name = ['050', '005', '100', '010']

m050_roc_sample_loader = DataLoader(
    val_file_list[0],
    batch_size=500,
    drop_last=False,
    shuffle=True,
    num_workers=1,
)
loaders_list.append(m050_roc_sample_loader)


m005_roc_sample_loader = DataLoader(
    val_file_list[1],
    batch_size=500,
    drop_last=False,
    shuffle=True,
    num_workers=1,
)
loaders_list.append(m005_roc_sample_loader)


m100_roc_sample_loader = DataLoader(
    val_file_list[2],
    batch_size=500,
    drop_last=False,
    shuffle=True,
    num_workers=1,
)
loaders_list.append(m100_roc_sample_loader)


m010_roc_sample_loader = DataLoader(
    val_file_list[3],
    batch_size=500,
    drop_last=False,
    shuffle=True,
    num_workers=1,
)
loaders_list.append(m010_roc_sample_loader)

In [52]:
OUTDIR = '/standard/ldmxuva/gnn_files/note_stuff/eat_equiv_eot_results/'

for i, loader in enumerate(loaders_list):
    y_scores_dynamic = []
    y_scores_static  = []
    y_true           = []
    event_numbers    = []
    file_numbers     = []

    print('beginning inference for signal mass ' + loader_name[i])
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
    
            out_dynamic = model_preloaded_dynamic(data)
            out_static  = model_preloaded_static(data)
    
            probs_dynamic = torch.softmax(out_dynamic, dim=1)[:, 1]
            probs_static  = torch.softmax(out_static,  dim=1)[:, 1]
    
            y_scores_dynamic.append(probs_dynamic.cpu())
            y_scores_static.append(probs_static.cpu())
            y_true.append(data.y.cpu())
            event_numbers.append(data.event_number.cpu())
            file_numbers.append(data.file_number.cpu())
    
            #ensure cleanup
            del data, out_dynamic, out_static, probs_dynamic, probs_static 
    
    #now save out to a .npz
    y_true = torch.cat(y_true).numpy()
    y_scores_static = torch.cat(y_scores_static).numpy()
    y_scores_dynamic = torch.cat(y_scores_dynamic).numpy()
    file_numbers = torch.cat(file_numbers).numpy()
    event_numbers = torch.cat(event_numbers).numpy()

    
    combined_arr = np.column_stack((y_scores_dynamic, y_scores_static, y_true, event_numbers, file_numbers))
    np.savez(OUTDIR + 'validation_inference_' + loader_name[i] + '.npz', inference_info = combined_arr)
    print('completed inference for signal mass ' + loader_name[i])
    
    
    
    

beginning inference for signal mass 050
completed inference for signal mass 050
beginning inference for signal mass 005
completed inference for signal mass 005
beginning inference for signal mass 100
completed inference for signal mass 100
beginning inference for signal mass 010
completed inference for signal mass 010


In [53]:
#Great, so this is the efficient way to do the inference! Saved off the signal data. Should target some subset of the finished graphs to test out a similar loop
BACKGROUND_DIRECTORY = '/standard/ldmxuva/gnn_files/note_stuff/eat_equiv_eot_graphs/processed/'
background_paths = glob.glob(os.path.join(BACKGROUND_DIRECTORY, "e*"))

In [54]:
i = 0
file_list = []
while i < len(background_paths):

    #this block runs our inference
    if (i % 100 == 0) and (i > 0):
        #identification
        print("we should create a data loader with these files")
        print(len(file_list))

        #create loader
        reform_list = [item for sublist in file_list for item in sublist]
        loader = DataLoader(reform_list, batch_size = 500, drop_last = False, shuffle=True, num_workers=1)

        #run inference, these get reset every time we hit this step.
        y_scores_dynamic = []
        y_scores_static  = []
        y_true           = []
        event_numbers    = []
        file_numbers     = []

        print(f'beginning inference on bunch {i}')
        with torch.no_grad():
            for data in loader:
                data = data.to(device)
        
                out_dynamic = model_preloaded_dynamic(data)
                out_static  = model_preloaded_static(data)
        
                probs_dynamic = torch.softmax(out_dynamic, dim=1)[:, 1]
                probs_static  = torch.softmax(out_static,  dim=1)[:, 1]
        
                y_scores_dynamic.append(probs_dynamic.cpu())
                y_scores_static.append(probs_static.cpu())
                y_true.append(data.y.cpu())
                event_numbers.append(data.event_number.cpu())
                file_numbers.append(data.file_number.cpu())
        
                #ensure cleanup
                del data, out_dynamic, out_static, probs_dynamic, probs_static

        #save off our stuff. 
        y_true = torch.cat(y_true).numpy()
        y_scores_static = torch.cat(y_scores_static).numpy()
        y_scores_dynamic = torch.cat(y_scores_dynamic).numpy()
        file_numbers = torch.cat(file_numbers).numpy()
        event_numbers = torch.cat(event_numbers).numpy()

        combined_arr = np.column_stack((y_scores_dynamic, y_scores_static, y_true, event_numbers, file_numbers))
        np.savez(OUTDIR + 'bkgd_inference_' + str(i) + '.npz', inference_info = combined_arr)
        print(f"Saved inference results for bunch {i}")

        #clean stuff up.
        del y_true, y_scores_static, y_scores_dynamic, file_numbers, event_numbers, loader
        file_list.clear()
        reform_list.clear()

        #step
        i += 1


    #this block steps through and loads for all other states
    else:  
        file_list.append(torch.load(background_paths[i], weights_only = False))
        i += 1


#now the cleanup, for non interval of 100 finish.
reform_list = [item for sublist in file_list for item in sublist]
loader = DataLoader(reform_list, batch_size = 500, drop_last = False, shuffle=True, num_workers=1)
#run inference, these get reset every time we hit this step.
y_scores_dynamic = []
y_scores_static  = []
y_true           = []
event_numbers    = []
file_numbers     = []

print(f'beginning inference on bunch {i}')
with torch.no_grad():
    for data in loader:
        data = data.to(device)
        
        out_dynamic = model_preloaded_dynamic(data)
        out_static  = model_preloaded_static(data)
        
        probs_dynamic = torch.softmax(out_dynamic, dim=1)[:, 1]
        probs_static  = torch.softmax(out_static,  dim=1)[:, 1]
        
        y_scores_dynamic.append(probs_dynamic.cpu())
        y_scores_static.append(probs_static.cpu())
        y_true.append(data.y.cpu())
        event_numbers.append(data.event_number.cpu())
        file_numbers.append(data.file_number.cpu())
        
        #ensure cleanup
        del data, out_dynamic, out_static, probs_dynamic, probs_static

#save off our stuff. 
y_true = torch.cat(y_true).numpy()
y_scores_static = torch.cat(y_scores_static).numpy()
y_scores_dynamic = torch.cat(y_scores_dynamic).numpy()
file_numbers = torch.cat(file_numbers).numpy()
event_numbers = torch.cat(event_numbers).numpy()

combined_arr = np.column_stack((y_scores_dynamic, y_scores_static, y_true, event_numbers, file_numbers))
np.savez(OUTDIR + 'bkgd_inference_' + str(i) + '.npz', inference_info = combined_arr)
print(f"Saved inference results for bunch {i}")

#clean stuff up.
del y_true, y_scores_static, y_scores_dynamic, file_numbers, event_numbers, loader
file_list.clear()
reform_list.clear()

we should create a data loader with these files
100
beginning inference on bunch 100
Saved inference results for bunch 100
we should create a data loader with these files
99
beginning inference on bunch 200
Saved inference results for bunch 200
we should create a data loader with these files
99
beginning inference on bunch 300


KeyboardInterrupt: 

In [38]:
arr1 = (np.load('/standard/ldmxuva/gnn_files/note_stuff/eat_equiv_eot_results/bkgd_inference_100.npz'))['inference_info']
arr2 = (np.load('/standard/ldmxuva/gnn_files/note_stuff/eat_equiv_eot_results/bkgd_inference_200.npz'))['inference_info']